Analyze season difference of low-cost sensors.

Time period is 03012022~03012023.

Data readings are averaged every 60 minutes.

If missing time frame is smaller than 10% of the total length, interpolation is applied. Otherwise, that sensor is dropped.

The whole map is separated into a 15x15 grid.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Data Preprocessing

In [3]:
def clean_timestamp(df):
    # df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_convert("US/Pacific")
    ts = pd.to_datetime(df["timestamp"], format="mixed")
    df["year"] = ts.dt.year
    df["month"] = ts.dt.month
    df["day"] = ts.dt.day
    df["hour"] = ts.dt.hour
    df["minute"] = ts.dt.minute

    df = df.drop(columns=["timestamp"])
    return df

def average_60min(df, col_name="pm25"):
    df = df[["year", "month", "day", "hour", col_name]]
    df = df.groupby(["year", "month", "day", "hour"]).mean().reset_index()
    return df

In [4]:
date_range = pd.date_range(start="2022-03-01-00-00-00",
                          end="2023-02-28-23-00-00",
                          freq="60min")
date_range = pd.DataFrame(date_range, columns=["timestamp"])
date_range = clean_timestamp(date_range)
date_range

,year,month,day,hour,minute
0,2022,3,1,0,0
1,2022,3,1,1,0
2,2022,3,1,2,0
3,2022,3,1,3,0
4,2022,3,1,4,0
...,...,...,...,...,...
8755,2023,2,28,19,0
8756,2023,2,28,20,0
8757,2023,2,28,21,0
8758,2023,2,28,22,0


In [13]:
dir_path = "../SJVAir_Data/raw/03012022_03012023/purpleair/"
for file_name in os.listdir(dir_path):
    if not file_name.endswith(".csv"):
        continue

    df = pd.read_csv(dir_path + file_name)

    # if missing data is more than 10%, then discard
    if len(df) < 0.8 * len(date_range) * 60:
        print(len(df), "  discard before processing")
        continue

    # discard if ab channels' difference is greater than 100
    try:
        diff = np.mean(df["pm25"][0::2].values) - np.mean(df["pm25"][1::2].values)
        if np.abs(diff) > 100:
            print(len(df), "  discard due to channel difference")
    except:
        continue

    df = average_60min(clean_timestamp(df))

    if len(df) < 0.8 * len(date_range):
        # if missing data is more than 10%, then discard
        print(len(df), "  discard after processing")
        continue

    df = pd.merge(date_range, df, how="left",
                  on=["year", "month", "day", "hour"])
    df["pm25"] = df["pm25"].interpolate(method="linear")
    df.to_csv("./data/processed/purpleair/{}".format(file_name), index=False)

26926   discard before processing
0   discard before processing
0   discard before processing
313737   discard before processing
378867   discard before processing
26926   discard before processing
341633   discard before processing
0   discard before processing
292089   discard before processing
0   discard before processing
1177   discard before processing
0   discard before processing
0   discard before processing
0   discard before processing
0   discard before processing
0   discard before processing
0   discard before processing
407164   discard before processing
148024   discard before processing
0   discard before processing


In [14]:
# add location
fresno_sensor = pd.read_csv("../SJVAir_Data/fresno_sensors.csv")

dir_path = "./data/processed/purpleair/"
for file_name in os.listdir(dir_path):
    if not file_name.endswith(".csv"):
        continue

    sensor_id = file_name.split(".")[0]
    df = pd.read_csv(dir_path + file_name)
    sensor_loc = fresno_sensor[fresno_sensor["id"] == sensor_id][["longitude", "latitude"]].to_numpy()[0]
    df["longitude"] = sensor_loc[0]
    df["latitude"] = sensor_loc[1]
    df.to_csv(dir_path + file_name, index=False)